# DTU BTech Major Project (GP159) — Llama-3.2-1B-Instruct Evaluation
### Delhi Technological University | Department of Computer Science & Engineering | Session 2026–27
**Assigned Investigator:** **Faqre Alam** (`23/CS/150`)
**Project Supervisor:** Ms. Gull Kaur, Assistant Professor, CSE, DTU
**Target Model Regime:** `Llama-3.2-1B-Instruct` (Evaluating FP16, 8-bit & 4-bit Quantization)

---
### ⚡ 1-Click Instructions for Kaggle:
1. In the right panel under **Session options**:
   - Set **Accelerator** to **GPU T4 x1** (16 GB VRAM).
   - Turn **Internet** toggle to **ON**.
2. Click **Run All** (or run each cell sequentially from top to bottom).
3. When finished, download your result zip file from the bottom of the notebook!


In [ ]:
# 1. Verify NVIDIA T4 GPU Environment
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")


In [ ]:
# 2. Clone repository & pull latest updates
import os
if not os.path.exists("run_experiments.py"):
    if os.path.exists("final-year-btp"):
        %cd final-year-btp
    else:
        !git clone https://github.com/itsalam149/final-year-btp.git
        %cd final-year-btp

# Pull latest code and OOM fixes from GitHub
if os.path.exists(".git"):
    !git pull origin main


In [ ]:
# 3. Install Evaluation Dependencies (Pinned versions for 100% stability)
!pip install -q "datasets==2.21.0" "huggingface-hub<0.29.0" lm-eval==0.4.3
!pip install -q accelerate bitsandbytes optimum langdetect immutabledict gptqmodel tabulate


In [ ]:
# 4. Authenticate Hugging Face (Required for Meta Llama-3.2 gated model)
import os
import getpass
from huggingface_hub import login, HfApi

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    try:
        hf_token = getpass.getpass("Enter your HF Token (or press enter to use cached token): ").strip()
    except Exception:
        pass
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Hugging Face Authentication Successful!")
else:
    print("ℹ️ Checking local login cache...")


In [ ]:
# 5. Execute Full Evaluation Suite for Llama-3.2-1B-Instruct
# Runs FP16 baseline, 8-bit & 4-bit variants across all 7 tasks
# Automatically persists results incrementally and skips completed tasks
!python run_experiments.py --model llama-3.2-1b --skip-completed


In [ ]:
# 6. Display Completed Results for Llama-3.2-1B-Instruct
import pandas as pd
if os.path.exists("results/results_master.csv"):
    df = pd.read_csv("results/results_master.csv")
    model_df = df[df["model"] == "llama-3.2-1b"]
    display(model_df)


In [ ]:
# 7. Archive Results into a Single Zip File for Download
!zip -r faqre_llama_results.zip results/
print("\n✅ SUCCESS! Download faqre_llama_results.zip from the right panel under Data -> Output.")
